# Islands and interior voids — 2D conformal mesh

`confMesh2dGMSH` treats an inner polygon that lies in another grain as an
**island** (its own ELSET, carved as a hole in the parent). An inner hole
that is **not** a grain is an **interior void** (no mesh, no ELSET).

`island_cover_frac` (default 0.95) is the area fraction of the inner polygon
that must lie in the outer polygon.

Requires `gmsh` (`pip install upxo[mesh]`). `confMesh2d` (pygmsh) is deprecated.
Canonical mesh-only demo: `confMesh2d_gmsh.ipynb`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import box

from upxo.meshing.gsmesh2d import mesh_gs
from upxo.meshing.writer_ABQ import summarize_inp


## Island grain (inclusion)

In [ ]:
island = {
    1: box(0, 0, 3, 3).difference(box(1, 1, 2, 2)),  # matrix
    2: box(1, 1, 2, 2),  # island
}
ri = mesh_gs(island, mesh_size_gb=0.25, mesh_size_bulk=0.5,
             mesh_algo=6, recombine_to_quads=False)
mi = ri['mesher']
mi.form_elsets_gmsh(); mi.build_boundary_nsets(); mi.build_gb_nset()
print({k: len(v) for k, v in mi.elsets.items()})
print(mi.validation_report)
fig, ax = mi.plot_by_grain(figsize=(5, 5), show_gb=True, show_nsets=True,
                           title='island grain.2 inside grain.1')
fig

## Interior void (not a grain)

In [ ]:
voids = {1: box(0, 0, 3, 3).difference(box(1, 1, 2, 2))}
rv = mesh_gs(voids, mesh_size_gb=0.25, mesh_size_bulk=0.5,
             mesh_algo=6, recombine_to_quads=False)
mv = rv['mesher']
mv.form_elsets_gmsh(); mv.build_boundary_nsets(); mv.build_gb_nset()
print('elsets', list(mv.elsets))
print(mv.validation_report)
fig, ax = mv.plot_by_grain(figsize=(5, 5), show_gb=True, show_nsets=True,
                           title='void in grain.1 (no island ELSET)')
fig

In [ ]:
out = Path.cwd() / 'confMesh2d_islands_voids_out'
p1 = mi.export_abaqus_inp(out / 'island_cps3.inp', plane='stress')
p2 = mv.export_abaqus_inp(out / 'void_cps3.inp', plane='stress')
p1, summarize_inp(p1), p2, summarize_inp(p2)